In [1]:
import os, time, math
from dataclasses import dataclass
from typing import List, Dict, Tuple, Set

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from rdkit import Chem
from rdkit.Chem import Descriptors, Crippen, QED
from rdkit import RDLogger
RDLogger.DisableLog("rdApp.*")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)


Device: cuda


In [2]:
DATA_PATH = "./cache_moses/moses_splits_vocab.pt"
obj = torch.load(DATA_PATH)

train_sm, val_sm, test_sm = obj["splits"]
stoi, itos = obj["stoi"], obj["itos"]
cfg0 = obj.get("cfg", {})

PAD, BOS, EOS = "<PAD>", "<BOS>", "<EOS>"
pad_id = stoi[PAD]
bos_id = stoi[BOS]
eos_id = stoi[EOS]
vocab_size = len(itos)

MAX_LEN = int(cfg0.get("MAX_LEN", 140))  
print("Loaded:", len(train_sm), len(val_sm), len(test_sm), "Vocab:", vocab_size, "MAX_LEN:", MAX_LEN)

Loaded: 200000 20000 20000 Vocab: 29 MAX_LEN: 140


In [3]:
def encode(sm: str, stoi: Dict[str,int], max_len: int) -> List[int]:
    ids = [stoi[BOS]] + [stoi[c] for c in sm if c in stoi] + [stoi[EOS]]
    if len(ids) < max_len:
        ids = ids + [stoi[PAD]] * (max_len - len(ids))
    else:
        ids = ids[:max_len]
        ids[-1] = stoi[EOS]
    return ids

In [4]:
class SmilesVAEDataset(Dataset):
    def __init__(self, smiles_list: List[str], stoi: Dict[str,int], max_len: int):
        self.smiles = smiles_list
        self.stoi = stoi
        self.max_len = max_len

    def __len__(self):
        return len(self.smiles)

    def __getitem__(self, idx):
        ids = encode(self.smiles[idx], self.stoi, self.max_len)
        return torch.tensor(ids, dtype=torch.long)

train_ds = SmilesVAEDataset(train_sm, stoi, MAX_LEN)
val_ds   = SmilesVAEDataset(val_sm, stoi, MAX_LEN)

BATCH_SIZE = 256 if DEVICE=="cuda" else 128
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=(DEVICE=="cuda"), drop_last=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=(DEVICE=="cuda"))

print("Batches:", len(train_loader), len(val_loader))

Batches: 781 79


In [5]:
@dataclass
class VAEConfig:
    vocab_size: int
    emb_dim: int = 256
    enc_hidden: int = 512
    dec_hidden: int = 512
    num_layers: int = 1
    z_dim: int = 64
    dropout: float = 0.0

class SmilesVAE(nn.Module):
    def __init__(self, cfg: VAEConfig, pad_id: int, bos_id: int, eos_id: int):
        super().__init__()
        self.cfg = cfg
        self.pad_id = pad_id
        self.bos_id = bos_id
        self.eos_id = eos_id

        self.embed = nn.Embedding(cfg.vocab_size, cfg.emb_dim, padding_idx=pad_id)

        self.enc_gru = nn.GRU(cfg.emb_dim, cfg.enc_hidden, num_layers=cfg.num_layers,
                              batch_first=True, dropout=cfg.dropout if cfg.num_layers > 1 else 0.0)

        self.to_mu = nn.Linear(cfg.enc_hidden, cfg.z_dim)
        self.to_logvar = nn.Linear(cfg.enc_hidden, cfg.z_dim)

        self.z_to_h0 = nn.Linear(cfg.z_dim, cfg.dec_hidden * cfg.num_layers)

        self.dec_gru = nn.GRU(cfg.emb_dim, cfg.dec_hidden, num_layers=cfg.num_layers,
                              batch_first=True, dropout=cfg.dropout if cfg.num_layers > 1 else 0.0)

        self.fc_out = nn.Linear(cfg.dec_hidden, cfg.vocab_size)

    def encode(self, x):
        emb = self.embed(x)
        _, h = self.enc_gru(emb)
        h_last = h[-1]
        return self.to_mu(h_last), self.to_logvar(h_last)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + std * eps

    def decode(self, z, x_in):
        B = z.size(0)
        L = self.cfg.num_layers
        h0 = self.z_to_h0(z).view(L, B, self.cfg.dec_hidden).contiguous()
        emb = self.embed(x_in)
        out, _ = self.dec_gru(emb, h0)
        return self.fc_out(out)

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        x_in = x[:, :-1]
        x_tgt = x[:, 1:]
        logits = self.decode(z, x_in)
        return logits, x_tgt, mu, logvar

In [6]:
def kl_divergence(mu, logvar):
    return 0.5 * torch.sum(torch.exp(logvar) + mu**2 - 1.0 - logvar, dim=1)

def recon_loss(logits, targets, pad_id: int):
    B, T, V = logits.shape
    return F.cross_entropy(logits.reshape(B*T, V), targets.reshape(B*T), ignore_index=pad_id)

def run_epoch(vae: SmilesVAE, loader, optimizer=None, kl_weight: float = 1.0):
    train = optimizer is not None
    vae.train(train)
    total = {"loss": 0.0, "recon": 0.0, "kl": 0.0}
    n = 0
    for x in loader:
        x = x.to(DEVICE, non_blocking=True)
        logits, tgt, mu, logvar = vae(x)
        r = recon_loss(logits, tgt, pad_id)
        kl = kl_divergence(mu, logvar).mean()
        loss = r + kl_weight * kl

        if train:
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(vae.parameters(), 1.0)
            optimizer.step()

        total["loss"] += loss.item()
        total["recon"] += r.item()
        total["kl"] += kl.item()
        n += 1

    for k in total:
        total[k] /= max(1, n)
    return total

In [7]:
@torch.no_grad()
def sample_vae(vae: SmilesVAE, n: int, max_len: int, temperature: float = 0.9, batch_size: int = 512) -> List[str]:
    vae.eval()
    samples = []

    remaining = n
    while remaining > 0:
        B = min(batch_size, remaining)
        remaining -= B

        z = torch.randn(B, vae.cfg.z_dim, device=DEVICE)

        L = vae.cfg.num_layers
        h = vae.z_to_h0(z).view(L, B, vae.cfg.dec_hidden).contiguous()

        cur = torch.full((B, 1), bos_id, dtype=torch.long, device=DEVICE)
        finished = torch.zeros(B, dtype=torch.bool, device=DEVICE)
        out_ids = [[] for _ in range(B)]

        for _ in range(max_len - 1):
            emb = vae.embed(cur)
            out, h = vae.dec_gru(emb, h)
            logits = vae.fc_out(out[:, -1, :]) / max(1e-8, temperature)
            probs = F.softmax(logits, dim=-1)

            nxt = torch.multinomial(probs, 1).squeeze(1)  # (B,)
            nxt = torch.where(finished, torch.tensor(eos_id, device=DEVICE), nxt)

            nxt_cpu = nxt.tolist()
            fin_cpu = finished.tolist()
            for i, nid in enumerate(nxt_cpu):
                if fin_cpu[i]:
                    continue
                if nid == eos_id:
                    finished[i] = True
                elif nid not in (pad_id, bos_id):
                    out_ids[i].append(nid)

            cur = nxt.unsqueeze(1)

            if finished.all():
                break

        samples.extend(["".join(itos[j] for j in ids) for ids in out_ids])

    return samples

In [8]:
def to_mol(smiles: str):
    return Chem.MolFromSmiles(smiles)

def canonical(smiles: str) -> str | None:
    m = to_mol(smiles)
    if m is None:
        return None
    return Chem.MolToSmiles(m, canonical=True)

train_canon: Set[str] = set()
for s in train_sm:
    cs = canonical(s)
    if cs is not None:
        train_canon.add(cs)

def validity(smiles_list: List[str]) -> Tuple[float, List[str]]:
    valid_canon = []
    for s in smiles_list:
        cs = canonical(s)
        if cs is not None:
            valid_canon.append(cs)
    return len(valid_canon) / max(1, len(smiles_list)), valid_canon

def uniqueness(valid_canon: List[str]) -> float:
    return 0.0 if len(valid_canon)==0 else len(set(valid_canon))/len(valid_canon)

def novelty(valid_canon: List[str], train_set: Set[str]) -> float:
    return 0.0 if len(valid_canon)==0 else sum(1 for s in valid_canon if s not in train_set)/len(valid_canon)

def novelty_unique(valid_canon: List[str], train_set: Set[str]) -> float:
    uv = set(valid_canon)
    return 0.0 if len(uv)==0 else sum(1 for s in uv if s not in train_set)/len(uv)

def compute_properties(valid_canon: List[str]) -> Dict[str,float]:
    if len(valid_canon)==0:
        return {"mw_mean": math.nan, "logp_mean": math.nan, "qed_mean": math.nan}
    mws, logps, qeds = [], [], []
    for s in valid_canon:
        m = to_mol(s)
        if m is None:
            continue
        mws.append(Descriptors.MolWt(m))
        logps.append(Crippen.MolLogP(m))
        qeds.append(QED.qed(m))
    if len(mws)==0:
        return {"mw_mean": math.nan, "logp_mean": math.nan, "qed_mean": math.nan}
    return {"mw_mean": sum(mws)/len(mws), "logp_mean": sum(logps)/len(logps), "qed_mean": sum(qeds)/len(qeds)}

def evaluate_smiles(samples: List[str], train_set: Set[str]) -> Dict[str,float]:
    v, valid_canon = validity(samples)
    out = {
        "n_samples": len(samples),
        "validity": v,
        "n_valid": len(valid_canon),
        "uniqueness": uniqueness(valid_canon),
        "novelty": novelty(valid_canon, train_set),
        "novelty_unique": novelty_unique(valid_canon, train_set),
    }
    out.update(compute_properties(valid_canon))
    return out

In [10]:
Z_LIST = [8, 32, 128] 
EPOCHS = 10                  
N_SAMPLES = 5000
TEMP = 0.9

LR = 3e-4
WEIGHT_DECAY = 1e-2

results = []
os.makedirs("./checkpoints_moses", exist_ok=True)

for zdim in Z_LIST:
    print("\n" + "="*70)
    print(f"Training MOSES VAE with z_dim={zdim}")

    cfg = VAEConfig(vocab_size=vocab_size, z_dim=zdim)
    vae = SmilesVAE(cfg, pad_id, bos_id, eos_id).to(DEVICE)
    optimizer = torch.optim.AdamW(vae.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    history = []
    for epoch in range(1, EPOCHS+1):
        kl_w = min(1.0, epoch / max(1, EPOCHS//2))
        t0 = time.time()
        tr = run_epoch(vae, train_loader, optimizer=optimizer, kl_weight=kl_w)
        va = run_epoch(vae, val_loader, optimizer=None, kl_weight=kl_w)
        t1 = time.time()

        print(f"Epoch {epoch:02d} | kl_w={kl_w:.2f} | "
              f"train L={tr['loss']:.3f} (R={tr['recon']:.3f}, KL={tr['kl']:.3f}) | "
              f"val L={va['loss']:.3f} (R={va['recon']:.3f}, KL={va['kl']:.3f}) | {t1-t0:.1f}s")

        history.append({
            "epoch": epoch, "kl_w": kl_w,
            **{f"train_{k}": v for k,v in tr.items()},
            **{f"val_{k}": v for k,v in va.items()}
        })

    t0 = time.time()
    gen = sample_vae(vae, n=N_SAMPLES, max_len=MAX_LEN, temperature=TEMP, batch_size=512)
    t1 = time.time()

    m = evaluate_smiles(gen, train_canon)
    gen_seconds = t1 - t0
    m["gen_seconds"] = gen_seconds
    m["samples_per_sec"] = N_SAMPLES / gen_seconds
    m["valid_per_sec"] = m["n_valid"] / gen_seconds

    m["z_dim"] = zdim
    m["final_train_recon"] = history[-1]["train_recon"]
    m["final_train_kl"] = history[-1]["train_kl"]
    m["final_val_recon"] = history[-1]["val_recon"]
    m["final_val_kl"] = history[-1]["val_kl"]

    results.append(m)

    ckpt_path = f"./checkpoints_moses/vae_moses_z{zdim}.pt"
    torch.save({
        "model_state": vae.state_dict(),
        "cfg": cfg.__dict__,
        "stoi": stoi,
        "itos": itos,
        "max_len": MAX_LEN
    }, ckpt_path)

    hist_path = f"./checkpoints_moses/vae_moses_z{zdim}_history.csv"
    pd.DataFrame(history).to_csv(hist_path, index=False)

    print("Eval metrics:", m)
    print("Saved:", ckpt_path)
    print("History:", hist_path)

df = pd.DataFrame(results).sort_values("z_dim").reset_index(drop=True)
df


Training MOSES VAE with z_dim=8
Epoch 01 | kl_w=0.20 | train L=0.923 (R=0.923, KL=0.000) | val L=0.715 (R=0.715, KL=0.000) | 26.7s
Epoch 02 | kl_w=0.40 | train L=0.682 (R=0.682, KL=0.000) | val L=0.655 (R=0.655, KL=0.000) | 26.9s
Epoch 03 | kl_w=0.60 | train L=0.641 (R=0.641, KL=0.000) | val L=0.628 (R=0.628, KL=0.000) | 27.0s
Epoch 04 | kl_w=0.80 | train L=0.620 (R=0.620, KL=0.000) | val L=0.612 (R=0.612, KL=0.000) | 26.9s
Epoch 05 | kl_w=1.00 | train L=0.605 (R=0.605, KL=0.000) | val L=0.601 (R=0.601, KL=0.000) | 27.1s
Epoch 06 | kl_w=1.00 | train L=0.595 (R=0.595, KL=0.000) | val L=0.592 (R=0.592, KL=0.000) | 27.1s
Epoch 07 | kl_w=1.00 | train L=0.586 (R=0.586, KL=0.000) | val L=0.584 (R=0.584, KL=0.000) | 27.0s
Epoch 08 | kl_w=1.00 | train L=0.579 (R=0.579, KL=0.000) | val L=0.579 (R=0.579, KL=0.000) | 27.0s
Epoch 09 | kl_w=1.00 | train L=0.573 (R=0.573, KL=0.000) | val L=0.574 (R=0.574, KL=0.000) | 26.9s
Epoch 10 | kl_w=1.00 | train L=0.567 (R=0.567, KL=0.000) | val L=0.570 (R=0.

,n_samples,validity,n_valid,uniqueness,novelty,novelty_unique,mw_mean,logp_mean,qed_mean,gen_seconds,samples_per_sec,valid_per_sec,z_dim,final_train_recon,final_train_kl,final_val_recon,final_val_kl
0,5000,0.9236,4618,0.998917,0.985492,0.985476,304.669716,2.449554,0.811804,0.355077,14081.461089,13005.637462,8,0.567005,1.853879e-07,0.570483,6.260460e-08
1,5000,0.9242,4621,0.999351,0.989180,0.989389,302.202193,2.502006,0.806690,0.319787,15635.406074,14450.242293,32,0.567257,4.124570e-07,0.570012,3.460945e-07
2,5000,0.9238,4619,0.999567,0.989608,0.989604,306.744933,2.514861,0.807119,0.326628,15307.935827,14141.471117,128,0.565152,9.725209e-07,0.567723,8.782518e-07
